<a href="https://colab.research.google.com/github/Towa-1103/Experiment/blob/main/res01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 必要なツールのインストール
!pip install -q bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4bit量子化の設定（VRAMを節約）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Tokenizer ロード中...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("7Bモデル（4bit）ロード中...（3分程度）")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("ロード完了！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.6 MB/s eta 0:00:00
Tokenizer ロード中...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

ロード完了！


In [10]:
# モデルがメモリ上に存在して動くかテスト
inputs = tokenizer("おい", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

おい附着の改善についての提案 | ブ


In [23]:
# memories.json を空リストでリセット
with open("memories.json", "w", encoding="utf-8") as f:
  f.write("[]")

In [24]:
import os

# 必ず /content を基準にする
%cd /content

REPO_URL = "https://github.com/Towa-1103/Experiment.git"
TARGET_DIR = "/content/Experiment"

if not os.path.exists(TARGET_DIR):
  !git clone {REPO_URL}
  %cd {TARGET_DIR}
else:
  %cd {TARGET_DIR}
  !git pull

print("\n 現在地:", os.getcwd())
print("\n リポジトリの同期が完了しました。")

/content
/content/Experiment
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.24 KiB | 1.24 MiB/s, done.
From https://github.com/Towa-1103/Experiment
   4f69fab..1ed448a  main       -> origin/main
Updating 4f69fab..1ed448a
Fast-forward
 dialogue_processor.py | 48 ++++++++++++++++++++++--------------------------
 1 file changed, 22 insertions(+), 26 deletions(-)

 現在地: /content/Experiment

 リポジトリの同期が完了しました。


In [25]:
import glob
import importlib
import os
import dialogue_processor
import memory_manager

# 最新コードの反映
importlib.reload(dialogue_processor)
importlib.reload(memory_manager)

from dialogue_processor import extract_memories_from_log
from memory_manager import MemoryManager

# --- 設定 ---
LOG_DIR = "./LINE_talk"
MEMORY_FILE = "memories.json"
MIN_IMPORTANCE = 3

# memories.json を最初から作り直したい場合はコメントアウトを解除
# with open(MEMORY_FILE, "w", encoding="utf-8") as f:
#     f.write("[]")

manager = MemoryManager(MEMORY_FILE)
print(f"処理前の総記憶数: {len(manager.get_all())} 件")

# txtファイルを名前順に取得
target_files = sorted(glob.glob(os.path.join(LOG_DIR, "*.txt")))
print(f"処理対象ファイル: {len(target_files)} 件\n")

# --- ループ処理 ---
for idx, file_path in enumerate(target_files, 1):
  file_name = os.path.basename(file_path)
  print(f"[{idx}/{len(target_files)}] 処理中: {file_name}")

  try:
    with open(file_path, "r", encoding="utf-8") as f:
      raw_log = f.read()

    # 中身が空の場合はスキップ
    if not raw_log.strip():
      print("  -> スキップ（空ファイル）")
      continue

    # 記憶抽出
    extracted = extract_memories_from_log(
        raw_log, tokenizer, model, min_importance=MIN_IMPORTANCE
    )
    print(f"  -> 抽出候補: {len(extracted)} 件")

    # 保存（重複は自動で除外される）
    added = manager.add_memories(extracted)
    print(
        f"  -> 新規保存: {added} 件（現在の合計: {len(manager.get_all())} 件）"
    )

  except Exception as e:
    print(f"  -> エラー発生のためスキップ: {e}")

print("\n=== すべてのファイル処理が完了しました ===")
print(f"最終記憶数: {len(manager.get_all())} 件")

処理前の総記憶数: 0 件
処理対象ファイル: 24 件

[1/24] 処理中: Friend_A_202607.txt
  -> 抽出候補: 1 件
  -> 新規保存: 1 件（現在の合計: 1 件）
[2/24] 処理中: Friend_A_202608.txt
  -> 抽出候補: 7 件
  -> 新規保存: 1 件（現在の合計: 2 件）
[3/24] 処理中: Friend_A_202609.txt
  -> 抽出候補: 0 件
  -> 新規保存: 0 件（現在の合計: 2 件）
[4/24] 処理中: Friend_A_251125.txt
  -> 抽出候補: 4 件
  -> 新規保存: 1 件（現在の合計: 3 件）
[5/24] 処理中: Friend_A_251203.txt
  -> 抽出候補: 2 件
  -> 新規保存: 1 件（現在の合計: 4 件）
[6/24] 処理中: Friend_A_260401.txt
  -> 抽出候補: 3 件
  -> 新規保存: 1 件（現在の合計: 5 件）
[7/24] 処理中: Friend_A_260406.txt
  -> 抽出候補: 7 件
  -> 新規保存: 1 件（現在の合計: 6 件）
[8/24] 処理中: Friend_A_260408.txt
  -> 抽出候補: 1 件
  -> 新規保存: 1 件（現在の合計: 7 件）
[9/24] 処理中: Friend_A_260409.txt
  -> 抽出候補: 3 件
  -> 新規保存: 1 件（現在の合計: 8 件）
[10/24] 処理中: Friend_A_260608.txt
  -> 抽出候補: 4 件
  -> 新規保存: 1 件（現在の合計: 9 件）
[11/24] 処理中: Friend_A_260609.txt
  -> 抽出候補: 7 件
  -> 新規保存: 1 件（現在の合計: 10 件）
[12/24] 処理中: Friend_A_260610.txt
  -> 抽出候補: 3 件
  -> 新規保存: 1 件（現在の合計: 11 件）
[13/24] 処理中: Friend_A_260611.txt
  -> 抽出候補: 5 件
  -> 新規保存: 1 件（現在の合計: 12 件）
[

In [ ]:
import torch

# 1. テキストファイルの中身確認
with open("Friend_A_2025.txt", "r", encoding="utf-8") as f:
  raw_log = f.read()

print("=== [Friend_A_2025.txt の中身（先頭200文字）] ===")
print(raw_log[:200])
print("==========================================\n")

# 2. LLMの生の出力確認
import dialogue_processor
from dialogue_processor import EXTRACT_ALL_PROMPT

prompt = EXTRACT_ALL_PROMPT.format(dialogue=raw_log.strip())
messages = [
    {
        "role": "system",
        "content": (
            "あなたは対話ログから有益な記憶を網羅的に抽出し、正確なJSON配列のみを出力するデータ処理エンジンです。"
        ),
    },
    {"role": "user", "content": prompt},
]
text_input = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(text_input, return_tensors="pt").to(model.device)

with torch.no_grad():
  outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)

raw_response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
)

print("=== [LLMの生の返答] ===")
print(raw_response)
print("=======================")

In [14]:
# logs.zip を Experiment フォルダ配下に解凍する場合
!unzip -q /content/Experiment/LINE_talk.zip -d /content/Experiment/